# Análisis de Logs y Detección de Cuellos de Botella con PySpark

Iván López

### Objetivo
Procesar logs simulados de microservicios y detectar anomalías de rendimiento usando PySpark.

## Instrucciones
1. Genera archivos de log simulados (CSV) con datos de timestamp, service, endpoint, response_time_ms, status_code.
2. Carga los archivos con Spark.
3. Limpia los datos (nulos, tipos incorrectos).
4. Calcula KPIs: tiempo promedio, desviación estándar y porcentaje de errores.
5. Detecta outliers en response_time_ms mediante z-score.
6. Guarda los resultados procesados en formato Parquet.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, avg, round, desc, count, sum, when, lit, stddev, abs
from pyspark.sql.window import Window

## Sesión Spark

In [2]:
# Crear sesión Spark
# ToDo: inicializa SparkSession
try:
    sc.stop()
except Exception:
    pass

In [3]:
spark = (SparkSession.builder
         .appName("NotebookSession")
         .master("local[*]")
         .config("spark.ui.port", "0")
         .getOrCreate())

sc = spark.sparkContext
sc.setLogLevel("WARN")

print(spark.version, sc.appName)

4.0.1 NotebookSession


## Carga de datos

In [4]:
# Cargar logs con Spark
# ToDo: usa spark.read.csv con header=True
df_logs1 = spark.read.csv('logs_day1.csv', header=True) 
df_logs2 = spark.read.csv('logs_day2.csv', header=True) 
df_logs3 = spark.read.csv('logs_day3.csv', header=True) 
df_logs_raw = df_logs1.union(df_logs2).union(df_logs3)

In [5]:
df_logs_raw.show(10)
print(df_logs_raw.count(), 'logs')

+-------------------+--------+-----------+----------------+-----------+
|          timestamp| service|   endpoint|response_time_ms|status_code|
+-------------------+--------+-----------+----------------+-----------+
|2025-11-01 00:00:00| catalog|   /details|             306|        200|
|2025-11-01 00:00:10|  orders|   /details|             120|        200|
|2025-11-01 00:00:20|  orders|   /details|             234|        200|
|2025-11-01 00:00:30|  orders|     /items|             238|        200|
|2025-11-01 00:00:40|payments|     /login|             489|        500|
|2025-11-01 00:00:50|  orders|/send-email|             289|        200|
|2025-11-01 00:01:00|    auth|     /items|              81|        500|
|2025-11-01 00:01:10| catalog|     /items|             281|        200|
|2025-11-01 00:01:20|  orders|   /details|             281|        200|
|2025-11-01 00:01:30|payments|   /details|             255|        200|
+-------------------+--------+-----------+----------------+-----

In [6]:
df_logs_raw.printSchema()

root
 |-- timestamp: string (nullable = true)
 |-- service: string (nullable = true)
 |-- endpoint: string (nullable = true)
 |-- response_time_ms: string (nullable = true)
 |-- status_code: string (nullable = true)



## Procesamiento

In [7]:
# Limpiar y transformar datos
# ToDo: convertir columnas a tipo adecuado y filtrar valores inválidos
#Limpiar registros nulos o con status_code inválido
df_logs = df_logs_raw.dropna().filter(
    (col("status_code") >= 200) & (col("status_code") < 600)
)
#Convertir el timestamp a tipo timestamp.
df_logs = df_logs.withColumn("timestamp",to_timestamp(col("timestamp")))

In [8]:
df_logs.show(10)
print(df_logs.count(), 'logs')

+-------------------+--------+-----------+----------------+-----------+
|          timestamp| service|   endpoint|response_time_ms|status_code|
+-------------------+--------+-----------+----------------+-----------+
|2025-11-01 00:00:00| catalog|   /details|             306|        200|
|2025-11-01 00:00:10|  orders|   /details|             120|        200|
|2025-11-01 00:00:20|  orders|   /details|             234|        200|
|2025-11-01 00:00:30|  orders|     /items|             238|        200|
|2025-11-01 00:00:40|payments|     /login|             489|        500|
|2025-11-01 00:00:50|  orders|/send-email|             289|        200|
|2025-11-01 00:01:00|    auth|     /items|              81|        500|
|2025-11-01 00:01:10| catalog|     /items|             281|        200|
|2025-11-01 00:01:20|  orders|   /details|             281|        200|
|2025-11-01 00:01:30|payments|   /details|             255|        200|
+-------------------+--------+-----------+----------------+-----

In [9]:
df_logs.printSchema()

root
 |-- timestamp: timestamp (nullable = true)
 |-- service: string (nullable = true)
 |-- endpoint: string (nullable = true)
 |-- response_time_ms: string (nullable = true)
 |-- status_code: string (nullable = true)



## Análisis

In [10]:
# Calcular KPIs
# ToDo: agrupa por service y endpoint y calcula métricas

### Tiempo promedio de respuesta por servicio y endpoint

In [11]:
#tiempo promedio de respuesta por servicio y endpoint
df_svep_rt_avg=df_logs.groupBy("service", "endpoint").agg(round(avg("response_time_ms"), 2).alias("avg_response_time_ms")).orderBy(desc("avg_response_time_ms"))
df_svep_rt_avg.show(25)

+-------------+-----------+--------------------+
|      service|   endpoint|avg_response_time_ms|
+-------------+-----------+--------------------+
|       orders|   /details|              290.14|
|notifications|/send-email|              283.65|
|       orders|/send-email|              279.04|
|         auth|     /login|              278.48|
|         auth|   /details|              277.88|
|     payments|     /items|              276.48|
|         auth|     /items|              274.96|
|      catalog|     /login|              274.69|
|notifications|     /login|              274.54|
|       orders|     /items|              273.63|
|      catalog|  /checkout|              273.58|
|notifications|     /items|              271.96|
|     payments|   /details|              270.51|
|         auth|/send-email|              269.59|
|      catalog|/send-email|              268.04|
|notifications|  /checkout|              266.26|
|      catalog|     /items|              265.95|
|       orders|  /ch

In [12]:
#tiempo promedio de respuesta por servicio 
df_serv_rt_avg=df_logs.groupBy("service").agg(round(avg("response_time_ms"), 2).alias("avg_response_time_ms")).orderBy(desc("avg_response_time_ms"))
df_serv_rt_avg.show()

+-------------+--------------------+
|      service|avg_response_time_ms|
+-------------+--------------------+
|       orders|               272.7|
|         auth|              272.24|
|notifications|              271.43|
|      catalog|              267.44|
|     payments|              265.26|
+-------------+--------------------+



In [13]:
#tiempo promedio de respuesta por endpoint
df_endp_rt_avg=df_logs.groupBy("endpoint").agg(round(avg("response_time_ms"), 2).alias("avg_response_time_ms")).orderBy(desc("avg_response_time_ms"))
df_endp_rt_avg.show()

+-----------+--------------------+
|   endpoint|avg_response_time_ms|
+-----------+--------------------+
|     /items|              272.72|
|/send-email|              271.76|
|   /details|              271.32|
|     /login|              269.19|
|  /checkout|              264.04|
+-----------+--------------------+



### KPIs globales

In [14]:
#Promedio y desviación estándar del tiempo de respuesta
glob_kpis=df_logs.agg(
    round(avg("response_time_ms"), 2).alias("global_avg_rtime_ms"), 
    round(stddev("response_time_ms"), 2).alias("global_stddev_rtime_ms")
)
glob_kpis.show()

+-------------------+----------------------+
|global_avg_rtime_ms|global_stddev_rtime_ms|
+-------------------+----------------------+
|             269.81|                172.54|
+-------------------+----------------------+



In [15]:
#Porcentaje de errores 
glob_error_perc=df_logs.agg(
    count(lit(1)).alias("total_logs"),
    sum(when(col("status_code") >= 500, 1).otherwise(0)).alias("error_logs"),
    round((col("error_logs") / col("total_logs")) * 100, 2).alias("error_percentage")
)
glob_error_perc.show()

+----------+----------+----------------+
|total_logs|error_logs|error_percentage|
+----------+----------+----------------+
|      9000|       188|            2.09|
+----------+----------+----------------+



### Endpoints más lentos y más propensos a fallar

In [16]:
#Endpoint más lentos (3)
df_endp_rt_avg.show(3)

+-----------+--------------------+
|   endpoint|avg_response_time_ms|
+-----------+--------------------+
|     /items|              272.72|
|/send-email|              271.76|
|   /details|              271.32|
+-----------+--------------------+
only showing top 3 rows


In [17]:
#Endpoints más propensos a fallar (3)
endp_error_perc=df_logs.groupBy("endpoint").agg(
    count(lit(1)).alias("total_logs"),
    sum(when(col("status_code") >= 500, 1).otherwise(0)).alias("error_logs"),
    round((col("error_logs") / col("total_logs")) * 100, 2).alias("error_percentage")
).orderBy(desc("error_percentage"))
endp_error_perc.show(3)

+--------+----------+----------+----------------+
|endpoint|total_logs|error_logs|error_percentage|
+--------+----------+----------+----------------+
|  /login|      1734|        48|            2.77|
|/details|      1859|        41|            2.21|
|  /items|      1757|        37|            2.11|
+--------+----------+----------+----------------+
only showing top 3 rows


### Anomalías de rendimiento

In [18]:
# Detectar anomalías por z-score
# ToDo: usa PySpark.sql.functions
df_anomalies=df_logs.crossJoin(glob_kpis).withColumn(
    "z_score",
    round((col("response_time_ms") - col("global_avg_rtime_ms")) / col("global_stddev_rtime_ms"), 2)
).filter(abs(col("z_score")) > 3).select(
    "timestamp", 
    "service", 
    "endpoint", 
    "response_time_ms",  
    "status_code",
    "z_score"
).orderBy(desc("z_score"))
df_anomalies.show()
print(df_anomalies.count(), 'anomalías de rendimiento')

+-------------------+-------------+-----------+----------------+-----------+-------+
|          timestamp|      service|   endpoint|response_time_ms|status_code|z_score|
+-------------------+-------------+-----------+----------------+-----------+-------+
|2025-11-01 02:59:40|      catalog|  /checkout|            2450|        200|  12.64|
|2025-11-03 03:00:20|         auth|     /login|            2315|        200|  11.85|
|2025-11-03 01:57:30|       orders|     /items|            2235|        200|  11.39|
|2025-11-02 06:16:20|       orders|/send-email|            2155|        200|  10.93|
|2025-11-02 05:06:20|      catalog|/send-email|            2075|        200|  10.46|
|2025-11-02 02:49:50|notifications|  /checkout|            2055|        200|  10.35|
|2025-11-01 01:59:30|         auth|     /items|            2030|        200|   10.2|
|2025-11-02 02:55:30|       orders|/send-email|            2005|        200|  10.06|
|2025-11-03 04:47:40|notifications|     /login|            1995| 

In [19]:
#usando ventanas 
windows = Window.partitionBy("service", "endpoint")
df_anomalies_windows = df_logs.withColumn(
    "avg_rtime_w", 
    avg(col("response_time_ms")).over(windows)
).withColumn(
    "stddev_rtime_w", 
    stddev(col("response_time_ms")).over(windows)
).withColumn(
    "z_score",
    round((col("response_time_ms") - col("avg_rtime_w")) / col("stddev_rtime_w"), 2)
).filter(abs(col("z_score")) > 3).select(
    "timestamp", 
    "service", 
    "endpoint", 
    "response_time_ms",  
    "status_code",
    "z_score"
).orderBy(desc("z_score"))
df_anomalies_windows.show()
print(df_anomalies_windows.count(), 'anomalías de rendimiento')

+-------------------+-------------+-----------+----------------+-----------+-------+
|          timestamp|      service|   endpoint|response_time_ms|status_code|z_score|
+-------------------+-------------+-----------+----------------+-----------+-------+
|2025-11-02 06:13:20|     payments|  /checkout|            1940|        200|  11.76|
|2025-11-02 03:40:30|notifications|   /details|            1925|        200|  11.21|
|2025-11-02 02:49:50|notifications|  /checkout|            2055|        200|  10.95|
|2025-11-01 02:59:40|      catalog|  /checkout|            2450|        200|  10.57|
|2025-11-01 05:30:20|     payments|/send-email|            1685|        200|  10.31|
|2025-11-03 03:00:20|         auth|     /login|            2315|        200|  10.17|
|2025-11-02 05:06:20|      catalog|/send-email|            2075|        200|  10.08|
|2025-11-02 03:13:00|      catalog|   /details|            1445|        200|  10.04|
|2025-11-01 07:06:40|       orders|     /login|            1625| 

## Guardar datasets

In [20]:
df_logs.write.mode("overwrite").parquet("../datasets/df_logs.parquet")
df_svep_rt_avg.write.mode("overwrite").parquet("../datasets/df_svep_rt_avg.parquet")
df_serv_rt_avg.write.mode("overwrite").parquet("../datasets/df_serv_rt_avg.parquet")
df_endp_rt_avg.write.mode("overwrite").parquet("../datasets/df_endp_rt_avg.parquet")
glob_kpis.write.mode("overwrite").parquet("../datasets/glob_kpis.parquet")
glob_error_perc.write.mode("overwrite").parquet("../datasets/glob_error_perc.parquet")
endp_error_perc.write.mode("overwrite").parquet("../datasets/endp_error_perc.parquet")
df_anomalies.write.mode("overwrite").parquet("../datasets/df_anomalies.parquet")
df_anomalies_windows.write.mode("overwrite").parquet("../datasets/df_anomalies_windows.parquet")